# Graphene transport in the ballistic regime
### A finite channel, three Dirac-point energies, and a conserved transverse momentum
**Professor Nano Labs · Nanoscale transport**

**Try it live in your browser:** [Graphene Ballistic Transport lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/graphene-ballistic-transport/)  
**Watch first:** [Ballistic electronic transport in graphene devices](https://www.youtube.com/watch?v=2RoP7nZv474)  
**Watch later:** [Deriving the Dirac Hamiltonian in Graphene](https://www.youtube.com/watch?v=TuWvO8oig0E) · [How electricity actually works at the nanoscale?](https://www.youtube.com/watch?v=z7dVABiYnTk)

We connect a graphene channel of length $L$ to semi-infinite graphene source and drain reservoirs. The three regions have independently chosen, constant **Dirac-point energies**. The channel is both **ballistic and phase coherent**; the two interfaces are retained in a single open-system calculation.

The notebook connects four views of the same device: fixed-$k_y$ band slices, the Dirac-point profile, transmission $T(E,k_y)$, and the energy-resolved local density of states $\rho(x,E;k_y)$. A live, offline browser laboratory is embedded near the end, with synchronized probe-energy sliders, playback, presets, and CSV export.

**Primary reference:** T. Low, S. Hong, J. Appenzeller, S. Datta, and M. Lundstrom, *Conductance asymmetry of graphene pn junction*, IEEE Transactions on Electron Devices **56**, 1292–1299 (2009), [arXiv:0811.1295](https://arxiv.org/abs/0811.1295), especially Sec. III. We use its transverse-mode NEGF idea and a gauge-equivalent nearest-neighbor mode chain. **This is a coherent finite-channel teaching extension, not a reproduction of the paper's single-junction conductance calculations.**

### Run this notebook
Use **Restart Kernel and Run All**. This copy is saved without outputs to keep the download small; running it rebuilds every plot. The Python calculations require only NumPy and Matplotlib. The interactive laboratory uses embedded JavaScript, not `ipywidgets`, Plotly, a server, or an internet connection. If the embedded laboratory is blank, trust and rerun this notebook, or open the generated `Graphene_Ballistic_Transport_Lab.html` in a browser. Static plots remain available independently of JavaScript.

In [ ]:
# Run once only if your Python environment is missing these packages:
# %pip install numpy matplotlib

%matplotlib inline
import json
import html
import time
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import HTML, display

## 1. What do the three controls mean?

We write the local Dirac-point energy as

$$E_D(x)=\begin{cases}
E_{D,S},&x<0,\\
E_{D,C},&0\le x<L,\\
E_{D,D},&x\ge L.
\end{cases}$$

The code calls these `dirac_source_eV`, `dirac_channel_eV`, and `dirac_drain_eV`, rather than $V_S,V_C,V_D$. Raising one of these numbers shifts **both** its conduction and valence bands upward. These are electron energies, not electrostatic voltages: an electric potential $\phi$ would contribute an energy $-e\phi$.

A reservoir chemical potential $\mu_S$ or $\mu_D$ is a separate quantity. In particular, changing $E_{D,D}$ does **not** automatically apply a source–drain bias. Transmission and LDOS can be computed without specifying reservoir occupations. At equilibrium one may take $\mu_S=\mu_D=0$; the default probe energy is also zero. The labels n/p in the explorer describe which band contains the **probe energy**, not a self-consistently calculated carrier density.

**Geometry assumptions:** the sheet is homogeneous and effectively wide in $y$, so transverse momentum is conserved and continuous. This is not a finite-width nanoribbon with an armchair or zigzag quantization condition. There is no self-consistent Poisson calculation, disorder, phonon scattering, or dephasing.

In [ ]:
from dataclasses import dataclass, replace
from typing import Optional
import numpy as np

HBAR_eVs = 6.582119569e-16
E_CHARGE_C = 1.602176634e-19
PLANCK_Js = 6.62607015e-34

@dataclass(frozen=True)
class Device:
    """Energies in eV, distances in nm; no self-consistent electrostatics."""
    dirac_source_eV: float = -0.20
    dirac_channel_eV: float = +0.20
    dirac_drain_eV: float = -0.20
    channel_length_nm: float = 40.0
    buffer_length_nm: float = 12.0
    hopping_eV: float = 2.7
    carbon_bond_nm: float = 0.142
    valley: int = +1
    eta_transmission_eV: float = 1.0e-9
    eta_ldos_eV: float = 1.0e-3

    def __post_init__(self):
        for name, value in vars(self).items():
            if not np.isfinite(value):
                raise ValueError(f"{name} must be finite.")
        if min(self.channel_length_nm, self.hopping_eV,
               self.carbon_bond_nm, self.eta_transmission_eV,
               self.eta_ldos_eV) <= 0:
            raise ValueError("Lengths, hopping, and broadenings must be positive.")
        if self.buffer_length_nm < 0 or self.valley not in (-1, +1):
            raise ValueError("buffer_length_nm >= 0; valley must be +1 or -1.")
        if self.n_channel_cells + 2*self.n_buffer_cells > 20000:
            raise ValueError("Device is too long for this teaching implementation.")

    @property
    def a0_nm(self):
        return np.sqrt(3.0)*self.carbon_bond_nm

    @property
    def cell_length_nm(self):
        return 1.5*self.carbon_bond_nm

    @property
    def hbar_vF_eVnm(self):
        return self.hopping_eV*self.cell_length_nm

    @property
    def vF_m_per_s(self):
        return self.hbar_vF_eVnm*1e-9/HBAR_eVs

    @property
    def n_channel_cells(self):
        return max(1, int(np.floor(self.channel_length_nm/self.cell_length_nm + 0.5)))

    @property
    def n_buffer_cells(self):
        return max(0, int(np.floor(self.buffer_length_nm/self.cell_length_nm + 0.5)))

    @property
    def actual_length_nm(self):
        return self.n_channel_cells*self.cell_length_nm

In [ ]:
# EDIT THESE INPUTS to change all subsequent static Python examples.
# The interactive browser laboratory has its own controls, initialized from this device.
device = Device(
    dirac_source_eV=-0.20,
    dirac_channel_eV=+0.20,
    dirac_drain_eV=-0.20,
    channel_length_nm=40.0,
    buffer_length_nm=12.0,       # Plotted lead sections, not additional barriers.
    hopping_eV=2.7,
    carbon_bond_nm=0.142,
    valley=+1,
    eta_transmission_eV=1e-9,    # Near-unitary ballistic transport.
    eta_ldos_eV=1e-3,           # Finite spectral resolution: 1 meV.
)
selected_ky_per_nm = 0.18
probe_energy_eV = 0.0
comparison_ky_per_nm = [0.0, 0.12, 0.18, 0.24]
energy_grid_eV = np.linspace(-0.45, 0.45, 901)  # 1 meV grid for static plots.

print(f"Requested L = {device.channel_length_nm:.3f} nm")
print(f"Actual L    = {device.actual_length_nm:.3f} nm ({device.n_channel_cells} A/B cells)")
print(f"Cell length = {device.cell_length_nm:.3f} nm")
print(f"hbar*vF     = {device.hbar_vF_eVnm:.4f} eV nm")
print(f"vF          = {device.vF_m_per_s:.3e} m/s")

The chosen $t=2.7$ eV and C–C bond distance $a_{cc}=0.142$ nm give $v_F=3ta_{cc}/(2\hbar)\approx0.874\times10^6$ m/s. **The band plots and transport use this same velocity; an independent $10^6$ m/s value is not silently substituted.** These are conventional teaching parameters for the nearest-neighbor model; see [Castro Neto et al., Rev. Mod. Phys. 81, 109 (2009)](https://arxiv.org/abs/0709.1163), Sec. II.

The length is rounded to complete two-orbital mode cells of length $a_x=3a_{cc}/2$. This is an **atomic tight-binding model**, not a finite-difference grid whose spacing should be refined independently of the hopping. The displayed length is the actual length used in the Hamiltonian.

## 2. The Dirac cone viewed one transverse momentum at a time

Throughout this notebook, **$k_y$ means momentum measured from the selected valley**, in nm$^{-1}$, not the absolute crystal wavevector. The longitudinal wavevector $q_x$ is likewise shifted to the valley origin.

For a uniform region $j$, the low-energy Dirac bands are

$$E_{j,\pm}(q_x;k_y)=E_{D,j}\pm\hbar v_F\sqrt{q_x^2+k_y^2}.$$

At $k_y=0$, a cut passes through the Dirac crossing. At nonzero $k_y$, the cut has a **projected mode gap**

$$\Delta_{\mathrm{mode}}=2\hbar v_F|k_y|.$$

This is not a true gap in the two-dimensional sheet: the $k_y=0$ cut still touches. For a chosen energy, the Dirac approximation gives

$$q_{x,j}^2=\left(\frac{E-E_{D,j}}{\hbar v_F}\right)^2-k_y^2.$$

Positive $q_{x,j}^2$ means propagation in that region; negative $q_{x,j}^2$ means an evanescent wave with $q_{x,j}=i\kappa_j$. A valence-band region is **not automatically forbidden**: it can carry propagating hole-like states. These are bulk band slices of the three uniform regions; finite-channel resonances are computed separately by NEGF.

In [ ]:
def make_geometry(device: Device):
    """Each A/B pair is displayed as one cell, at its center.

    The plotted source/drain buffers are parts of the ideal leads, not walls.
    Exact semi-infinite graphene self-energies attach at their outer ends.
    """
    ns, nc = device.n_buffer_cells, device.n_channel_cells
    cell_dirac = np.r_[np.full(ns, device.dirac_source_eV),
                       np.full(nc, device.dirac_channel_eV),
                       np.full(ns, device.dirac_drain_eV)]
    x_nm = (np.arange(cell_dirac.size)-ns+0.5)*device.cell_length_nm
    return x_nm, cell_dirac, np.repeat(cell_dirac, 2)


def mode_hopping(ky_per_nm: float, device: Device) -> float:
    """ky is a displacement from a chosen valley, NOT absolute crystal ky.

    In the paper: t_y=t[1+exp(i k_y^crystal a0)]. Remove its phase by a
    site-dependent unitary transformation of the open mode chain.
    Near K_y^crystal=valley*2*pi/(3*a0), |t_y| is the following positive b.
    """
    if not np.isfinite(ky_per_nm):
        raise ValueError("ky_per_nm must be finite.")
    # Deliberately restrict the teaching model to one valley neighborhood.
    if abs(ky_per_nm)*device.a0_nm > 0.5:
        raise ValueError("Use |ky|*a0 <= 0.5 in this valley-resolved notebook.")
    return 2*device.hopping_eV*np.cos(device.valley*np.pi/3 +
                                     ky_per_nm*device.a0_nm/2)


def band_offset(qx_per_nm, ky_per_nm: float, device: Device,
                model: str = "tight_binding"):
    """Positive energy measured from E_D; qx is shifted to the valley origin."""
    qx = np.asarray(qx_per_nm, dtype=float)
    if model == "dirac":
        return device.hbar_vF_eVnm*np.sqrt(qx*qx + ky_per_nm**2)
    if model != "tight_binding":
        raise ValueError("model must be 'tight_binding' or 'dirac'.")
    t, b = device.hopping_eV, mode_hopping(ky_per_nm, device)
    return np.sqrt((t-b)**2 + 4*t*b*np.sin(qx*device.cell_length_nm/2)**2)


def propagating(energies_eV, dirac_eV: float, t: float, b: float):
    """One open longitudinal mode per spin/valley, away from band edges.

    At ky=0 the two bands touch; E=E_D is the continuous, open-mode limit.
    At the edge of a nonzero fixed-ky gap the group velocity is zero.
    """
    eps = np.abs(np.asarray(energies_eV, dtype=float)-dirac_eV)
    gap = abs(t-b)
    lower_open = np.ones_like(eps, dtype=bool) if gap < 1e-12 else eps > gap+1e-12
    return lower_open & (eps < t+b-1e-12)

In [ ]:
qx_per_nm = np.linspace(-0.85, 0.85, 601)
fig, ax = plt.subplots(figsize=(9.2, 4.8), constrained_layout=True)
for ky in [0.0, 0.12, 0.24, 0.36]:
    e = band_offset(qx_per_nm, ky, device, model="dirac")
    # One line object contains both branches, so each ky uses one default color.
    ax.plot(np.r_[qx_per_nm, np.nan, qx_per_nm], np.r_[e, np.nan, -e],
            label=fr"$k_y={ky:.2f}$ nm$^{{-1}}$")
ax.set(xlabel=r"$q_x$ from the valley (nm$^{-1}$)", ylabel=r"$E-E_D$ (eV)",
       title="Dirac cone: different fixed transverse-momentum cuts")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.show()

## 3. The Hamiltonian actually used for NEGF

Rather than apply a naive centered finite difference to a single Dirac cone, we use graphene's **nearest-neighbor $p_z$ mode-space tight-binding Hamiltonian**. This avoids introducing an extra low-energy cone through that discretization.

For each conserved transverse mode, the two-orbital blocks in the reference can be unfolded into a scalar chain. The complex intercell hopping has magnitude

$$b(k_y)=|t_y|=2t\cos\!\left(\nu\frac{\pi}{3}+\frac{a_0 k_y}{2}\right),
\qquad a_0=\sqrt{3}a_{cc},\quad \nu=\pm1,$$

within the valley neighborhood used here. The relation to the paper's absolute crystal momentum is $k_y^{\rm crystal}=\nu\,2\pi/(3a_0)+k_y$. A site-dependent phase transformation removes the complex hopping phases in this open chain. An additional sign convention centers the longitudinal valley at $q_x=0$.

The scalar chain has alternating hoppings $-t,b,-t,b,\ldots$, with the same $E_D$ on the A and B orbitals in a cell. Its uniform-region Bloch Hamiltonian is

$$H_j(q_x;k_y)=\begin{pmatrix}
E_{D,j}&-t+b\,e^{-iq_xa_x}\\
-t+b\,e^{iq_xa_x}&E_{D,j}
\end{pmatrix},\qquad a_x=\frac{3a_{cc}}{2}.$$

Consequently, the **exact bands of the implemented model** are

$$E_{j,\pm}^{\rm TB}=E_{D,j}\pm
\sqrt{(t-b)^2+4tb\sin^2(q_xa_x/2)}.$$

The exact fixed-$k_y$ half-gap is $|t-b|$, which approaches $\hbar v_F|k_y|$ near a valley. The following band plot and the thresholds drawn on LDOS maps use these **tight-binding** bands, so their mode thresholds agree with the transport solver. Small lattice corrections are retained; exact continuum identities should be interpreted as low-energy limits.

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 4.8), constrained_layout=True)
offset = band_offset(qx_per_nm, selected_ky_per_nm, device)
for name, ed, ls in [("source", device.dirac_source_eV, "-"),
                     ("channel", device.dirac_channel_eV, "--"),
                     ("drain", device.dirac_drain_eV, ":")]:
    ax.plot(np.r_[qx_per_nm, np.nan, qx_per_nm],
            np.r_[ed+offset, np.nan, ed-offset], linestyle=ls,
            label=fr"{name}: $E_D={ed:+.2f}$ eV")
ax.axhline(probe_energy_eV, linestyle="-.", linewidth=1, label="probe energy")
ax.set(xlabel=r"$q_x$ from the valley (nm$^{-1}$)", ylabel="Energy (eV)",
       title=fr"Bands used by NEGF: $k_y={selected_ky_per_nm:.3f}$ nm$^{{-1}}$")
ax.legend(ncol=2)
ax.grid(alpha=.25)
plt.show()

## 4. NEGF in three equations

The finite computational region includes the channel and short, uniform source/drain buffers. Exact self-energies connect it to the **semi-infinite** reservoirs:

$$G^r(E,k_y)=\left[(E+i\eta)I-H(k_y)-\Sigma_S^r-\Sigma_D^r\right]^{-1}.$$

Define $\Gamma_j=i(\Sigma_j^r-\Sigma_j^{r\dagger})=-2\,\mathrm{Im}\Sigma_j^r$ on each terminal orbital. Then

$$\boxed{T(E,k_y)=\mathrm{Tr}\,[\Gamma_S G^r\Gamma_DG^a]
=\Gamma_S\Gamma_D|G^r_{1N}|^2.}$$

The local density of states on orbital $i$ is

$$\boxed{\rho_i(E,k_y)=-\frac{1}{\pi}\operatorname{Im}G^r_{ii}
=\frac{A_{ii}}{2\pi},\quad A=i(G^r-G^a).}$$

We display a cell-averaged line density

$$\rho(x_j,E;k_y)=\frac{\rho_{A_j}+\rho_{B_j}}{a_x},$$

in **states/(eV nm), per spin and one transverse valley mode**. It is not yet a two-dimensional DOS per unit area, and it is not occupied carrier density. A factor of four is **not** included in a single-mode transmission probability.

### Two numerical broadenings, two purposes
`eta_transmission_eV = 1e-9` approximates the unitary ballistic limit. `eta_ldos_eV = 1e-3` gives the spectral plots a 1 meV resolution. The LDOS setting is **not a scattering rate used to calculate $T$**. All LDOS Green functions and their lead self-energies are evaluated at the same complex energy. Broadening can produce tails in nominal gaps and spread bound-state peaks; reduce $\eta$ and refine the energy grid together when resolving narrow features.

### 4.1 Semi-infinite source and drain self-energies

The outward device–lead hopping is $b$. The successive inward lead hoppings are $t,b,t,\ldots$. If $z=E-E_{D,j}+i\eta$, the lead surface Green function satisfies

$$g=\frac{1}{z-\dfrac{t^2}{z-b^2g}},\qquad \Sigma=b^2g.$$

Thus $\Sigma$ solves

$$z\Sigma^2-(z^2+b^2-t^2)\Sigma+zb^2=0.$$

The code chooses the **retarded root**, with $\operatorname{Im}\Sigma\le0$, and evaluates the quadratic without subtractive cancellation. This analytic surface solution is possible for our uniform ideal leads. It replaces the iterative surface-Green-function construction in the more general numerical procedure of the reference; it is not a wide-band-contact approximation.

In [ ]:
def surface_self_energy(energies_eV, dirac_eV: float,
                        t: float, b: float, eta_eV: float):
    r"""Exact retarded self-energy of a semi-infinite alternating chain.

    The boundary hopping is b. The inward hoppings in the lead are t,b,t,...
    g = 1/[z - t^2/(z-b^2*g)], Sigma=b^2*g.
    Hence z*Sigma^2 - (z^2+b^2-t^2)*Sigma + z*b^2 = 0.
    Choose the causal root Im Sigma <= 0. Compute one root without
    cancellation and the other from the product Sigma_1*Sigma_2=b^2.
    No wide-band or constant-contact-DOS approximation is used.
    """
    if eta_eV <= 0 or t <= 0 or b <= 0:
        raise ValueError("eta, t, b must be positive.")
    z = np.atleast_1d(np.asarray(energies_eV, float)) - dirac_eV + 1j*eta_eV
    c = z*z + b*b - t*t
    root = np.sqrt(c*c - 4*b*b*z*z)
    numerator = np.where(np.abs(c+root) >= np.abs(c-root), c+root, c-root)
    s1 = numerator/(2*z)
    s2 = b*b/s1
    sigma = np.where(s1.imag <= s2.imag, s1, s2)
    if np.any(sigma.imag > 1e-10):
        raise FloatingPointError("A noncausal lead root was found.")
    return sigma

### 4.2 A recursive calculation instead of a large matrix inverse

For this scalar, tridiagonal mode chain, a left-to-right pass gives the left-connected Green functions. A right-to-left pass then obtains the **full** $G^r_{ii}$ needed for LDOS. The end-to-end propagator gives the Caroli transmission. The cost grows linearly with channel length for each energy and transverse mode.

The key lines are identified in the code. In particular, a left-connected Green function is **not** itself the full local Green function and should not be used directly as the LDOS.

In [ ]:
def recursive_green(energies_eV, onsite_eV, hoppings_eV,
                    sigma_source, sigma_drain, eta_eV: float,
                    diagonal: bool = True):
    r"""O(N_E*N_sites) scalar recursive Green function; no dense inversion.

    D_i = E+i eta-U_i-Sigma_S delta_i0-Sigma_D delta_i,N-1.
    Left-connected g_i=1/(D_i-|h_(i-1)|^2*g_(i-1)).
    G_1N = (product_i g_i)*(product_i h_i), evaluated in log-magnitude.
    A second, right-to-left pass obtains the FULL diagonal G_ii.
    """
    energies = np.atleast_1d(np.asarray(energies_eV, dtype=float))
    onsite = np.asarray(onsite_eV, dtype=float)
    hops = np.asarray(hoppings_eV, dtype=float)
    if energies.ndim != 1 or onsite.ndim != 1 or hops.size != onsite.size-1:
        raise ValueError("Use 1D arrays with len(hoppings)=len(onsite)-1.")
    if not all(np.all(np.isfinite(a)) for a in (energies, onsite, hops)):
        raise ValueError("The Green-function inputs must be finite.")
    if np.any(hops == 0) or eta_eV <= 0:
        raise ValueError("Use nonzero hoppings and positive eta.")
    nE, n = energies.size, onsite.size
    zbase = energies + 1j*eta_eV
    left_g = np.empty((nE, n), dtype=complex) if diagonal else None
    previous = np.zeros(nE, dtype=complex)
    log_G1N = np.zeros(nE)

    for i in range(n):
        d = zbase - onsite[i]
        if i == 0:
            d = d - sigma_source
        if i == n-1:
            d = d - sigma_drain
        left_self = hops[i-1]**2*previous if i else 0.0
        previous = 1.0/(d-left_self)
        log_G1N += np.log(np.abs(previous))
        if i:
            log_G1N += np.log(abs(hops[i-1]))
        if diagonal:
            left_g[:, i] = previous

    gamma_s = -2*np.asarray(sigma_source).imag
    gamma_d = -2*np.asarray(sigma_drain).imag
    log_T = (np.log(np.maximum(gamma_s, 1e-300)) +
             np.log(np.maximum(gamma_d, 1e-300)) + 2*log_G1N)
    transmission_raw = np.exp(np.clip(log_T, -745, 700))
    if not diagonal:
        return transmission_raw, None

    full_diagonal = np.empty_like(left_g)
    right_self = np.zeros(nE, dtype=complex)
    for i in range(n-1, -1, -1):
        d = zbase-onsite[i]
        if i == 0:
            d = d-sigma_source
        if i == n-1:
            d = d-sigma_drain
        left_self = hops[i-1]**2*left_g[:, i-1] if i else 0.0
        # This is the full open-system G_ii, not the left-connected g_i.
        full_diagonal[:, i] = 1.0/(d-left_self-right_self)
        right_g = 1.0/(d-right_self)
        right_self = hops[i-1]**2*right_g if i else 0.0
    return transmission_raw, full_diagonal

In [ ]:
def solve_mode(energies_eV, ky_per_nm: float, device: Device,
               quantity: str = "both"):
    """Return T(E,ky) and/or cell-averaged LDOS(E,x;ky).

    T is a probability per spin/valley (not multiplied by four).
    LDOS is in states/(eV nm), per spin and one transverse valley mode.
    Different eta values deliberately separate nearly unitary transport
    from finite-resolution spectral visualization.
    """
    if quantity not in ("both", "transmission", "ldos"):
        raise ValueError("quantity must be 'both', 'transmission', or 'ldos'.")
    energies = np.atleast_1d(np.asarray(energies_eV, dtype=float))
    x, cell_dirac, onsite = make_geometry(device)
    t, b = device.hopping_eV, mode_hopping(ky_per_nm, device)
    hops = np.tile([-t, b], len(x))[:-1]
    out = dict(energy_eV=energies, x_nm=x, dirac_profile_eV=cell_dirac,
               ky_per_nm=float(ky_per_nm), actual_length_nm=device.actual_length_nm,
               mode_half_gap_eV=abs(t-b))
    if quantity in ("both", "transmission"):
        eta = device.eta_transmission_eV
        ss = surface_self_energy(energies, device.dirac_source_eV, t, b, eta)
        sd = surface_self_energy(energies, device.dirac_drain_eV, t, b, eta)
        T, _ = recursive_green(energies, onsite, hops, ss, sd, eta, diagonal=False)
        both_leads_open = (propagating(energies, device.dirac_source_eV, t, b) &
                           propagating(energies, device.dirac_drain_eV, t, b))
        # Eliminate the infinitesimal eta-induced spectral tails in a closed lead.
        # An evanescent CHANNEL is not masked: its tunneling is kept exactly.
        T = np.where(both_leads_open, T, 0.0)
        if np.any(T > 1+2e-5) or np.any(T < 0):
            raise FloatingPointError("Transmission failed the single-mode bound.")
        out.update(transmission=T, source_open=propagating(
            energies, device.dirac_source_eV, t, b), drain_open=propagating(
            energies, device.dirac_drain_eV, t, b))
    if quantity in ("both", "ldos"):
        eta = device.eta_ldos_eV
        ss = surface_self_energy(energies, device.dirac_source_eV, t, b, eta)
        sd = surface_self_energy(energies, device.dirac_drain_eV, t, b, eta)
        _, gd = recursive_green(energies, onsite, hops, ss, sd, eta, diagonal=True)
        site_ldos = -gd.imag/np.pi
        if np.min(site_ldos) < -1e-8:
            raise FloatingPointError("The spectral density is not positive.")
        # Sum A+B in each cell, then divide by its physical length.
        cell_ldos = site_ldos.reshape(len(energies), len(x), 2).sum(axis=2)
        out["ldos_per_eV_nm"] = cell_ldos/device.cell_length_nm
    return out

## 5. One device, four complementary transport views

The source and drain each support a propagating mode only when

$$|t-b|<|E-E_{D,j}|<t+b,$$

with the continuous crossing limit included at $k_y=0$. At a nonzero-gap band edge, the longitudinal velocity vanishes. If either **lead** mode is closed, the plotted transmission is zero. In contrast, an evanescent **channel** is not removed: tunneling through its finite length is part of the NEGF solution.

In [ ]:
start = time.perf_counter()
selected_result = solve_mode(energy_grid_eV, selected_ky_per_nm, device)
transmission_curves = {
    ky: solve_mode(energy_grid_eV, ky, device, quantity="transmission")["transmission"]
    for ky in comparison_ky_per_nm
}
print(f"Computed {len(energy_grid_eV)} energies and {len(selected_result['x_nm'])} spatial cells.")
print(f"Elapsed time: {time.perf_counter()-start:.2f} s")
print(f"Selected-mode gap = {2*selected_result['mode_half_gap_eV']*1000:.2f} meV")
print(f"T(E0, ky) = {solve_mode([probe_energy_eV], selected_ky_per_nm, device, 'transmission')['transmission'][0]:.6f}")

In [ ]:
def add_interface_guides(ax, device):
    ax.axvline(0, linestyle=":", linewidth=1)
    ax.axvline(device.actual_length_nm, linestyle=":", linewidth=1)

x_nm, ed_cell, _ = make_geometry(device)
x_edges_nm = np.r_[x_nm-device.cell_length_nm/2, x_nm[-1]+device.cell_length_nm/2]
half_gap = selected_result["mode_half_gap_eV"]
fig, ax = plt.subplots(figsize=(9.4, 4.4), constrained_layout=True)
ax.stairs(ed_cell, x_edges_nm, baseline=None, label=r"Dirac-point energy $E_D(x)$", linewidth=2.2)
ax.stairs(ed_cell+half_gap, x_edges_nm, baseline=None, linestyle="--", label="upper mode threshold")
ax.stairs(ed_cell-half_gap, x_edges_nm, baseline=None, linestyle="--", label="lower mode threshold")
ax.axhline(probe_energy_eV, linestyle="-.", linewidth=1, label="probe energy")
add_interface_guides(ax, device)
ax.set(xlabel="Position x (nm)", ylabel="Energy (eV)", title="Source — finite channel — drain")
ax.legend(ncol=2)
ax.grid(alpha=.2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9.4, 4.6), constrained_layout=True)
for ky, T in transmission_curves.items():
    ax.plot(energy_grid_eV, T, label=fr"$k_y={ky:.2f}$ nm$^{{-1}}$")
ax.axvline(probe_energy_eV, linestyle=":", linewidth=1)
ax.set(xlabel="Energy E (eV)", ylabel="Single-mode transmission T", ylim=(-.02, 1.04),
       title=fr"Ballistic transmission through a {device.actual_length_nm:.3f} nm channel")
ax.legend(ncol=2)
ax.grid(alpha=.2)
plt.show()

**Interpretation.** At low energy, normal incidence approaches perfect Klein transmission through a scalar barrier. Oblique modes can reflect and develop interference resonances; see [Katsnelson, Novoselov, and Geim, Nature Physics 2, 620 (2006)](https://arxiv.org/abs/cond-mat/0604323). Here the underlying lattice model is retained: the $k_y=0$ curve can differ slightly from one at finite energy. The code does **not** artificially replace it with $T=1$.

### 5.1 The energy–position LDOS map

The same fixed color range is used for every LDOS map, so apparent brightness can be compared across $k_y$. Dark regions do not necessarily imply a true band gap; first compare with the projected mode thresholds. Bright channel features inside a lead's mode gap can be bound or weakly coupled states and need not carry a transmitted current.

In [ ]:
def plot_ldos(result, device, logarithmic=True):
    E, x = result["energy_eV"], result["x_nm"]
    de = E[1]-E[0] if len(E)>1 else .001
    extent = (x[0]-device.cell_length_nm/2, x[-1]+device.cell_length_nm/2,
              E[0]-de/2, E[-1]+de/2)
    fig, ax = plt.subplots(figsize=(9.4, 4.9), constrained_layout=True)
    kwargs = {"norm": LogNorm(vmin=1e-3, vmax=5)} if logarithmic else {"vmin":0,"vmax":5}
    image = ax.imshow(result["ldos_per_eV_nm"], origin="lower", aspect="auto",
                      extent=extent, interpolation="nearest", **kwargs)
    cb = fig.colorbar(image, ax=ax, extend="max")
    cb.set_label(r"LDOS [states/(eV nm)], one spin and valley mode")
    edges = np.r_[x-device.cell_length_nm/2, x[-1]+device.cell_length_nm/2]
    half = result["mode_half_gap_eV"]
    ax.stairs(result["dirac_profile_eV"]+half, edges, baseline=None, linestyle="--", linewidth=1.1)
    ax.stairs(result["dirac_profile_eV"]-half, edges, baseline=None, linestyle="--", linewidth=1.1)
    add_interface_guides(ax, device)
    ax.set(xlabel="Position x (nm)", ylabel="Energy E (eV)",
           title=fr"Energy-resolved LDOS: $k_y={result['ky_per_nm']:.3f}$ nm$^{{-1}}$, "
                 +fr"$\eta={device.eta_ldos_eV*1000:g}$ meV",
           ylim=(E[0], E[-1]))
    return fig, ax

plot_ldos(selected_result, device)
plt.show()

### 5.2 Compare LDOS at different transverse momenta

Each image below is a separate plot with identical spatial, energy, and color limits. The $k_y=0$ slice has no projected gap; larger $|k_y|$ separates its valence and conduction thresholds. Contact penetration and channel standing waves are found from the open-system Green function, not by drawing a local bulk DOS independently in each region.

In [ ]:
for ky in [0.0, 0.12, 0.24]:
    result = solve_mode(energy_grid_eV, ky, device, quantity="ldos")
    plot_ldos(result, device)
    plt.show()

### 5.3 A spatial line cut at one energy

An LDOS line cut can contain standing-wave oscillations. Its maxima indicate enhanced **available spectral weight** at that energy, not the peak of an occupied electron-density distribution. An actual density calculation would also require transverse-mode integration and appropriate occupations.

In [ ]:
fig, ax = plt.subplots(figsize=(9.4, 4.6), constrained_layout=True)
for ky in comparison_ky_per_nm:
    one = solve_mode([probe_energy_eV], ky, device, quantity="ldos")
    ax.plot(one["x_nm"], one["ldos_per_eV_nm"][0],
            label=fr"$k_y={ky:.2f}$ nm$^{{-1}}$")
add_interface_guides(ax, device)
ax.set(xlabel="Position x (nm)", ylabel=r"LDOS [states/(eV nm)]",
       title=fr"Spatial LDOS at $E_0={probe_energy_eV:+.3f}$ eV")
ax.legend(ncol=2)
ax.grid(alpha=.2)
plt.show()

### 5.4 Transmission across the energy–momentum plane

The dark wedges associated with closed source/drain modes differ from a finite-length channel's tunneling and interference structure. In the exact single-valley tight-binding model, small lattice corrections need not be even in the displacement $k_y$ within one valley. The time-reversal relation is $T_K(E,k_y)=T_{K'}(E,-k_y)$; the low-energy isotropic Dirac limit is simpler.

In [ ]:
ky_grid_per_nm = np.linspace(-0.50, 0.50, 101)
# A slightly coarser energy grid is sufficient for this overview map.
map_energy_eV = np.linspace(-0.45, 0.45, 451)
T_map = np.array([solve_mode(map_energy_eV, ky, device, "transmission")["transmission"]
                  for ky in ky_grid_per_nm])
fig, ax = plt.subplots(figsize=(9.4, 4.8), constrained_layout=True)
image = ax.imshow(T_map, origin="lower", aspect="auto", interpolation="nearest",
                  extent=(map_energy_eV[0], map_energy_eV[-1],
                          ky_grid_per_nm[0], ky_grid_per_nm[-1]), vmin=0, vmax=1)
fig.colorbar(image, ax=ax, label="Single-mode transmission T")
ax.set(xlabel="Energy E (eV)", ylabel=r"$k_y$ from the selected valley (nm$^{-1}$)",
       title="Transmission in the energy–transverse-momentum plane")
plt.show()

## 6. Checks before trusting the pictures

A colorful LDOS image is not a numerical validation. The tests below compare the recursive calculation with a direct matrix inverse, verify causal leads and positive LDOS, test a uniform device and closed lead modes, check invariance under a common energy shift and an artificial lead-buffer change, and compare the exact uniform LDOS with the one-dimensional band DOS.

We also check approach to an independent **continuum Dirac barrier result** for equal source and drain. If $u=(E-E_{D,S})/(\hbar v_F)$, $v=(E-E_{D,C})/(\hbar v_F)$, $k=\sqrt{u^2-k_y^2}$, and $q=\sqrt{v^2-k_y^2}$, continuity of the two-component Dirac spinor gives

$$T_D=\left[|\cos(qL)|^2+
\left|\frac{uv-k_y^2}{k}\frac{\sin(qL)}{q}\right|^2\right]^{-1}$$

when a lead mode is open. The $q\to0$ limit uses $\sin(qL)/q\to L$. This result is a benchmark only; all principal transmission and LDOS plots above are obtained from NEGF.

To approach the Dirac regime while keeping the barrier problem comparable, we lower all energies and $k_y$ by a scale factor and increase $L$ inversely. The lattice result should approach the continuum result, rather than being forced to agree exactly at finite energy.

In [ ]:
def continuum_barrier_transmission(energies_eV, ky_per_nm: float,
                                   device: Device):
    """Independent Dirac-wave-matching benchmark for equal source/drain.

    A useful low-energy comparison, not the solver used for the NEGF plots.
    Stable here for |Im(qx)*L| < 300; very large barriers can be set to zero.
    """
    if abs(device.dirac_source_eV-device.dirac_drain_eV) > 1e-12:
        raise ValueError("This benchmark requires equal source/drain Dirac energies.")
    energies = np.atleast_1d(np.asarray(energies_eV, dtype=float))
    u = (energies-device.dirac_source_eV)/device.hbar_vF_eVnm
    v = (energies-device.dirac_channel_eV)/device.hbar_vF_eVnm
    T = np.zeros_like(energies)
    valid = np.abs(u) > abs(ky_per_nm)+1e-13
    if abs(ky_per_nm) < 1e-13:
        return np.ones_like(energies)
    k = np.sqrt(u[valid]**2-ky_per_nm**2)
    q = np.sqrt((v[valid]**2-ky_per_nm**2).astype(complex))
    arg = q*device.actual_length_nm
    sincL = np.empty_like(q)
    tiny = np.abs(q) < 1e-12
    sincL[tiny] = device.actual_length_nm
    sincL[~tiny] = np.sin(arg[~tiny])/q[~tiny]
    denom = np.abs(np.cos(arg))**2 + np.abs(
        (u[valid]*v[valid]-ky_per_nm**2)*sincL/k)**2
    T[valid] = 1.0/denom
    return T

In [ ]:
def run_numerical_checks(verbose=True):
    """Independent algebraic, transport, spectral, and continuum-limit checks."""
    rows = []
    def record(name, error, tolerance):
        passed = np.isfinite(error) and error < tolerance
        rows.append((name, float(error), float(tolerance), passed))
        if verbose:
            print(f"{'PASS' if passed else 'FAIL':4s}  {name:44s}  error = {error:.3e}")
        if not passed:
            raise AssertionError(f"{name}: {error} >= {tolerance}")

    d = Device()
    # 1. A small device: compare recursive NEGF against a dense matrix inverse.
    small = replace(d, channel_length_nm=2.13, buffer_length_nm=0)
    E, ky = np.array([.137]), .14
    _, _, U = make_geometry(small)
    t, b = small.hopping_eV, mode_hopping(ky, small)
    h = np.tile([-t, b], len(U)//2)[:-1]
    ss = surface_self_energy(E, small.dirac_source_eV, t, b, .001)
    sd = surface_self_energy(E, small.dirac_drain_eV, t, b, .001)
    H = np.diag(U) + np.diag(h, 1) + np.diag(h, -1)
    M = (E[0]+1j*.001)*np.eye(len(U))-H
    M[0, 0] -= ss[0]; M[-1, -1] -= sd[0]
    dense = np.linalg.inv(M)
    tr, gd = recursive_green(E, U, h, ss, sd, .001)
    record("Recursive G_ii versus dense inverse", np.max(np.abs(gd[0]-dense.diagonal())), 1e-10)
    td = (-2*ss[0].imag)*(-2*sd[0].imag)*abs(dense[0, -1])**2
    record("Caroli T versus dense inverse", abs(tr[0]-td), 1e-10)

    # 2. The chosen surface root must satisfy the lead quadratic and causality.
    E = np.linspace(-.45, .45, 301)
    b = mode_hopping(.18, d); eta = 1e-6
    sig = surface_self_energy(E, d.dirac_source_eV, t, b, eta)
    z = E-d.dirac_source_eV+1j*eta
    residual = z*sig**2-(z*z+b*b-t*t)*sig+z*b*b
    scale = 1+np.abs(z*sig**2)+np.abs((z*z+b*b-t*t)*sig)+np.abs(z*b*b)
    record("Retarded lead quadratic residual", np.max(np.abs(residual)/scale), 1e-12)
    record("Lead causality violation", max(0., float(np.max(sig.imag))), 1e-12)

    # 3. A uniform sheet has unit transmission in every open mode.
    uniform = replace(d, dirac_source_eV=0, dirac_channel_eV=0, dirac_drain_eV=0)
    tu = solve_mode([-.34, -.28, .12, .23], .18, uniform, "transmission")["transmission"]
    record("Uniform graphene: T -> 1", np.max(np.abs(tu-1)), 5e-6)
    closed = solve_mode([0.], .18, uniform, "transmission")["transmission"]
    record("Closed lead mode: T = 0", np.max(np.abs(closed)), 1e-14)

    # 4. Full-spectrum unitarity and positivity, without forcibly clipping T to 1.
    out = solve_mode(E, .18, d)
    violation = max(0., float(np.max(out["transmission"])-1),
                    float(-np.min(out["transmission"])))
    record("Single-mode transmission bounds", violation, 2e-5)
    record("LDOS positivity violation", max(0., float(-np.min(out["ldos_per_eV_nm"]))), 1e-8)

    # 5. Zero-point energy changes cannot change the physics.
    delta = .073
    shifted = replace(d, dirac_source_eV=d.dirac_source_eV+delta,
                      dirac_channel_eV=d.dirac_channel_eV+delta,
                      dirac_drain_eV=d.dirac_drain_eV+delta)
    ea = np.array([-.35, .037, .36])
    before = solve_mode(ea, .18, d)
    after = solve_mode(ea+delta, .18, shifted)
    record("Common energy-shift invariance: T", np.max(np.abs(before["transmission"]-after["transmission"])), 1e-9)
    record("Common energy-shift invariance: LDOS", np.max(np.abs(before["ldos_per_eV_nm"]-after["ldos_per_eV_nm"])), 1e-8)

    # 6. Artificial plotting-buffer length cannot create physical reflections.
    no_pad = solve_mode(ea, .18, replace(d, buffer_length_nm=0), "transmission")
    record("Lead-buffer invariance (finite eta_T)", np.max(np.abs(before["transmission"]-no_pad["transmission"])), 2e-6)

    # 7. Reflection of the entire two-terminal device preserves transmission.
    asymmetric = replace(d, dirac_source_eV=-.23, dirac_channel_eV=-.04, dirac_drain_eV=.10)
    mirrored = replace(asymmetric, dirac_source_eV=asymmetric.dirac_drain_eV,
                       dirac_drain_eV=asymmetric.dirac_source_eV)
    ta = solve_mode([-.35, 0., .25], .09, asymmetric, "transmission")["transmission"]
    tb = solve_mode([-.35, 0., .25], .09, mirrored, "transmission")["transmission"]
    record("Source/drain interchange", np.max(np.abs(ta-tb)), 1e-9)
    tm = solve_mode(ea, -.18, replace(d, valley=-1), "transmission")["transmission"]
    record("Time-reversed valleys K,ky <-> K',-ky", np.max(np.abs(before["transmission"]-tm)), 1e-10)

    # 8. Spectral display broadening must not change the separate T calculation.
    broad = solve_mode(ea, .18, replace(d, eta_ldos_eV=.005), "transmission")
    record("LDOS eta leaves T unchanged", np.max(np.abs(before["transmission"]-broad["transmission"])), 1e-14)

    # 9. Analytic bulk DOS of this 1D band, including both signs of qx.
    exact_uniform = replace(uniform, eta_ldos_eV=1e-6)
    e = .2; b = mode_hopping(.18, exact_uniform)
    cq = (t*t+b*b-e*e)/(2*t*b)
    bulk_rho = abs(e)/(np.pi*exact_uniform.cell_length_nm*t*b*np.sqrt(1-cq*cq))
    rho = solve_mode([e], .18, exact_uniform, "ldos")["ldos_per_eV_nm"][0]
    record("Uniform LDOS versus analytic 1D DOS", np.max(np.abs(rho/bulk_rho-1)), 1e-6)

    # 10. Approach the continuum at fixed dimensionless barrier parameters.
    errors = []
    for scale in (1., .5, .25):
        ds = replace(d, dirac_source_eV=-.1*scale, dirac_channel_eV=.08*scale,
                     dirac_drain_eV=-.1*scale, channel_length_nm=20/scale)
        q = .04*scale
        tn = solve_mode([0], q, ds, "transmission")["transmission"][0]
        tc = continuum_barrier_transmission([0], q, ds)[0]
        errors.append(abs(tn-tc))
    record("Low-energy Dirac-limit error (last scale)", errors[-1], .003)
    if not (errors[2] < errors[1] < errors[0]):
        raise AssertionError("The tight-binding result did not approach the Dirac limit.")
    if verbose:
        print("\nDirac-limit errors on lowering E, U, ky and increasing L:",
              ", ".join(f"{e:.5f}" for e in errors))
        print(f"\nAll {len(rows)} numerical checks passed.")
    return rows


check_results = run_numerical_checks()

### A finite neutral channel still transmits evanescent modes

At $E=E_{D,C}$, a nonzero-$k_y$ channel mode is evanescent. For heavily doped ideal contacts, the continuum result approaches

$$T(k_y)\simeq \operatorname{sech}^2(|k_y|L).$$

This finite-length transport mechanism is discussed by [Tworzydło et al., Phys. Rev. Lett. 96, 246802 (2006)](https://arxiv.org/abs/cond-mat/0603315). With finite contact doping, the exact transmission includes additional contact matching, so the expression is an asymptotic benchmark, not a replacement for the lead self-energies.

In [ ]:
neutral_device = replace(device, dirac_source_eV=-0.30, dirac_channel_eV=0.0,
                         dirac_drain_eV=-0.30, channel_length_nm=40.0)
ky_test = np.linspace(0.0, 0.12, 81)
T_neutral = np.array([solve_mode([0.0], q, neutral_device, "transmission")["transmission"][0]
                      for q in ky_test])
T_heavy_contact_limit = 1/np.cosh(ky_test*neutral_device.actual_length_nm)**2
fig, ax = plt.subplots(figsize=(9.2, 4.4), constrained_layout=True)
ax.plot(ky_test, T_neutral, label="NEGF, finite graphene contact doping")
ax.plot(ky_test, T_heavy_contact_limit, "--", label=r"Continuum heavy-contact limit: $\mathrm{sech}^2(|k_y|L)$")
ax.set(xlabel=r"$k_y$ (nm$^{-1}$)", ylabel="Transmission at the channel Dirac energy",
       title="Evanescent transmission through a finite neutral channel", ylim=(-.02, 1.04))
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 7. Optional: from mode transmission to conductance

The primary plots are mode-resolved, not a conductance. For a wide sheet at zero temperature, with conserved continuous transverse momentum,

$$\frac{G}{W}=\frac{2e^2}{h}\sum_{\nu=K,K'}
\int\frac{dk_y}{2\pi}\,T_\nu(E_F,k_y).$$

The factor two here is spin; the valley sum is explicit. In the isotropic limit it is often written with a combined factor four. For a finite ribbon one instead needs its actual transverse boundary conditions and a mode sum; do not multiply by a second valley degeneracy if the ribbon modes already count it.

The helper below reports conductance per width in **S/µm** and refuses an integration window whose edge still carries an open lead mode. Converge both the $k_y$ range and grid when using it quantitatively. At finite temperature, further average over energy with $-\partial f/\partial E$.

For a finite-bias current, specify $\mu_S$ and $\mu_D$ independently and use the Landauer occupation difference. Distinct Dirac-point energies with $\mu_S=\mu_D$ still give **zero net equilibrium current**, even though their transmission and LDOS can be nontrivial.

In [ ]:
def conductance_per_width(EF_eV: float, device: Device,
                          ky_max_per_nm: float = 0.85, n_ky: int = 241):
    """Zero-temperature, wide-sheet conductance, in S per micrometer.

    Both valleys are evaluated, and a factor 2 accounts for spin.
    The chosen ky window must contain every propagating lead mode.
    This is not a finite-width armchair-ribbon quantization rule.
    """
    if n_ky < 5:
        raise ValueError("n_ky must be at least 5; converge the integration grid.")
    ky = np.linspace(-ky_max_per_nm, ky_max_per_nm, n_ky)
    total_integral = 0.0
    for valley in (+1, -1):
        d = replace(device, valley=valley)
        edge_b = [mode_hopping(q, d) for q in (ky[0], ky[-1])]
        if any(propagating([EF_eV], d.dirac_source_eV, d.hopping_eV, b)[0] and
               propagating([EF_eV], d.dirac_drain_eV, d.hopping_eV, b)[0]
               for b in edge_b):
            raise ValueError("Increase ky_max_per_nm: a lead mode reaches the integration edge.")
        vals = np.array([solve_mode([EF_eV], q, d, "transmission")["transmission"][0]
                         for q in ky])
        total_integral += np.sum(0.5*(vals[:-1]+vals[1:])*np.diff(ky))/(2*np.pi)
    # The ky integral has units 1/nm. 1000 converts to 1/micrometer.
    return 2*E_CHARGE_C**2/PLANCK_Js*total_integral*1000.0

In [ ]:
# Optional calculation; uncomment and converge n_ky as needed.
# g_per_width = conductance_per_width(0.0, device, ky_max_per_nm=0.85, n_ky=241)
# print(f"G/W = {g_per_width*1e6:.3f} microSiemens per micrometer")

## 8. Interactive laboratory — change the device and explore

The next cells embed a self-contained browser implementation of the **same recursive NEGF algorithm**. It evaluates the self-energies and Green functions on demand; the sliders do not merely switch between precomputed screenshots. The Python and JavaScript solvers were cross-checked on asymmetric devices, both valleys, and normal incidence.

The source, channel, and drain **Dirac-point energies**, channel length, selected $k_y$, valley, and LDOS broadening are adjustable. Changing the probe energy moves a cursor through the spectral map and recalculates the spatial LDOS line cut at the exact selected energy. A second synchronized energy slider sits directly above the spatial plot. **Sweep energy** plays this sequence automatically.

The browser map uses a 2 meV energy grid for responsiveness; the static Python examples above use 1 meV. Both use the same Hamiltonian. The browser's controls are independent of later Python variable changes: rerun its creation cell to reinitialize it from Python.

The following two implementation cells are long because the laboratory includes its own offline plotting and solver code. They can remain collapsed during a lecture.

In [ ]:
BROWSER_ENGINE_JS = r'''
// Exact, scalar mode-space NEGF. This is a direct JavaScript translation
// of solve_mode() in the notebook; no fitted or pretabulated transmission.
const GrapheneEngine = (() => {
  function csqrt(x,y) {
    const r=Math.hypot(x,y);
    if (x>=0) { const a=Math.sqrt((r+x)/2); return [a,a===0?0:y/(2*a)]; }
    const b=(y<0?-1:1)*Math.sqrt((r-x)/2);
    return [b===0?0:y/(2*b),b];
  }
  function cdiv(a,b,c,d) { const v=c*c+d*d; return [(a*c+b*d)/v,(b*c-a*d)/v]; }
  function sigma(E,U,t,b,eta) {
    const zr=E-U, zi=eta;
    const cr=zr*zr-zi*zi+b*b-t*t, ci=2*zr*zi;
    const rr=cr*cr-ci*ci-4*b*b*(zr*zr-zi*zi);
    const ri=2*cr*ci-8*b*b*zr*zi;
    const root=csqrt(rr,ri);
    const pr=cr+root[0], pi=ci+root[1], mr=cr-root[0], mi=ci-root[1];
    const big=(Math.hypot(pr,pi)>=Math.hypot(mr,mi))?[pr,pi]:[mr,mi];
    const s1=cdiv(big[0],big[1],2*zr,2*zi), s2=cdiv(b*b,0,s1[0],s1[1]);
    return s1[1]<=s2[1]?s1:s2;
  }
  function geometry(p) {
    const a=1.5*p.carbon_bond_nm;
    const nc=Math.max(1,Math.floor(p.channel_length_nm/a+0.5));
    const ns=Math.max(0,Math.floor(p.buffer_length_nm/a+0.5));
    const n=nc+2*ns, x=new Float64Array(n), U=new Float64Array(2*n);
    for(let j=0;j<n;j++) {
      x[j]=(j-ns+.5)*a;
      const u=j<ns?p.dirac_source_eV:(j<ns+nc?p.dirac_channel_eV:p.dirac_drain_eV);
      U[2*j]=u; U[2*j+1]=u;
    }
    return {a,nc,ns,n,x,U,L:nc*a,xmin:-ns*a,xmax:(nc+ns)*a};
  }
  function hopping(ky,p) {
    return 2*p.hopping_eV*Math.cos(p.valley*Math.PI/3+ky*Math.sqrt(3)*p.carbon_bond_nm/2);
  }
  function open(E,U,t,b) {
    const v=Math.abs(E-U), gap=Math.abs(t-b);
    return (gap<1e-12||v>gap+1e-12)&&v<t+b-1e-12;
  }
  function solve(energies,ky,p,eta,needLDOS=false) {
    const g=geometry(p), n=2*g.n, ne=energies.length, t=p.hopping_eV, b=hopping(ky,p);
    const T=new Float64Array(ne), rho=needLDOS?new Float64Array(ne*g.n):null;
    const leftR=needLDOS?new Float64Array(n):null;
    const leftI=needLDOS?new Float64Array(n):null;
    for(let ie=0;ie<ne;ie++) {
      const E=energies[ie], s=sigma(E,p.dirac_source_eV,t,b,eta);
      const d=sigma(E,p.dirac_drain_eV,t,b,eta);
      let prevR=0,prevI=0,logAmp=0;
      for(let j=0;j<n;j++) {
        let zr=E-g.U[j],zi=eta;
        if(j===0){zr-=s[0];zi-=s[1];}
        if(j===n-1){zr-=d[0];zi-=d[1];}
        const hp=j===0?0:((j-1)%2===0?t:b), h2=hp*hp;
        const dr=zr-h2*prevR,di=zi-h2*prevI,den=dr*dr+di*di;
        prevR=dr/den;prevI=-di/den;
        logAmp-=.5*Math.log(den);
        if(j>0)logAmp+=Math.log(hp);
        if(needLDOS){leftR[j]=prevR;leftI[j]=prevI;}
      }
      const logT=Math.log(Math.max(-2*s[1],1e-300))+Math.log(Math.max(-2*d[1],1e-300))+2*logAmp;
      T[ie]=(open(E,p.dirac_source_eV,t,b)&&open(E,p.dirac_drain_eV,t,b))?
        Math.exp(Math.max(-745,Math.min(700,logT))):0;
      if(needLDOS) {
        let rsr=0,rsi=0;
        for(let j=n-1;j>=0;j--) {
          let zr=E-g.U[j],zi=eta;
          if(j===0){zr-=s[0];zi-=s[1];}
          if(j===n-1){zr-=d[0];zi-=d[1];}
          const hp=j===0?0:((j-1)%2===0?t:b),h2=hp*hp;
          const lr=j>0?h2*leftR[j-1]:0,li=j>0?h2*leftI[j-1]:0;
          const dr=zr-lr-rsr,di=zi-li-rsi,den=dr*dr+di*di;
          rho[ie*g.n+Math.floor(j/2)]+=di/den/Math.PI/g.a;
          const rr=zr-rsr,ri=zi-rsi,rd=rr*rr+ri*ri;
          rsr=h2*rr/rd;rsi=-h2*ri/rd;
        }
      }
    }
    return {T,rho,geometry:g,halfGap:Math.abs(t-b),energies,ky};
  }
  return {sigma,geometry,hopping,open,solve};
})();
if(typeof module!=='undefined')module.exports=GrapheneEngine;

'''

In [ ]:
BROWSER_HTML_TEMPLATE = r'''
<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Graphene | Ballistic Transport Laboratory</title>
<style>
:root{--ink:#183041;--muted:#526574;--paper:#f3f6f8;--line:#dbe3e8;--accent:#176779;}
*{box-sizing:border-box} body{margin:0;background:var(--paper);color:var(--ink);font:15px/1.5 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}
main{max-width:1180px;margin:auto;padding:30px 24px 36px} h1{font-size:34px;letter-spacing:-.8px;line-height:1.15;margin:8px 0 10px} h2{font-size:19px;margin:0 0 5px;letter-spacing:-.3px} p{margin:6px 0 14px} .eyebrow{font-size:12px;font-weight:750;letter-spacing:1.8px;text-transform:uppercase;color:var(--accent)} .subtitle{font-size:16px;max-width:870px;color:var(--muted)}
.card{background:white;border:1px solid var(--line);border-radius:13px;padding:20px;margin-top:18px;box-shadow:0 2px 4px #152f4005} .small{font-size:13px;color:var(--muted)} .row{display:flex;align-items:center;flex-wrap:wrap;gap:10px 18px}.row.between{justify-content:space-between} .controls{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:15px 26px;margin-top:15px} .control label{display:flex;justify-content:space-between;gap:8px;font-size:13px;font-weight:650}.control output{font-variant-numeric:tabular-nums;color:var(--accent);white-space:nowrap} input[type=range]{width:100%;margin:10px 0 0;accent-color:var(--accent)} select,button{font:inherit;font-size:13px;border:1px solid #bdcdd5;border-radius:7px;padding:7px 10px;background:white;color:var(--ink)} button{cursor:pointer}button:hover{background:#edf4f6}button.primary{background:var(--accent);color:white;border-color:var(--accent)} .pill{border-radius:20px;background:#eaf2f4;font-size:12px;padding:5px 10px;font-weight:600} .stats{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:16px}.stat{border-left:3px solid var(--line);padding-left:13px}.stat span{display:block;font-size:11px;text-transform:uppercase;letter-spacing:.7px;color:var(--muted)}.stat strong{font-weight:650;font-size:18px;font-variant-numeric:tabular-nums}.grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px}.grid2 .card{min-width:0}.plot{width:100%;height:300px;display:block;margin-top:12px}.wideplot{height:325px}.ldosplot{height:380px}.legend{display:flex;flex-wrap:wrap;gap:8px 20px;font-size:12px;min-height:21px;margin-top:5px}.swatch{display:inline-block;width:21px;height:3px;vertical-align:middle;margin:0 6px 3px 0}.callout{background:#edf5f6;border-left:3px solid var(--accent);padding:10px 14px;margin-top:13px;font-size:13px}.error{color:#923929;font-weight:600}details{margin-top:15px}summary{cursor:pointer;font-weight:650}code{font-size:12px;background:#f0f3f5;padding:2px 4px;border-radius:3px}.footer{font-size:12px;color:var(--muted);margin:18px 0 0}.inline-control{display:flex;align-items:center;gap:8px;font-size:13px}.energyrow{display:flex;gap:18px;align-items:center;margin-top:12px}.energyrow input{flex:1;margin:0}.energyrow output{width:72px;white-space:nowrap;font-weight:650;font-variant-numeric:tabular-nums}a{color:var(--accent)} .status{min-height:22px;font-size:12px;margin-top:12px;color:var(--muted)}
@media(max-width:780px){main{padding:18px 12px}h1{font-size:28px}.controls{grid-template-columns:1fr 1fr}.grid2{grid-template-columns:1fr;gap:0}.stats{grid-template-columns:1fr 1fr}.card{padding:15px}.plot{height:300px}}
@media(max-width:480px){.controls{grid-template-columns:1fr}.stats{gap:8px}.stat strong{font-size:16px}.energyrow{gap:8px}}
</style></head><body><main>
<div class="eyebrow">Professor Nano · Computational semiconductor physics</div>
<h1>Graphene: ballistic transport</h1>
<p class="subtitle">Two semi-infinite reservoirs. One finite coherent channel. Change the three Dirac-point energies and connect the band slices, transmission, and local spectrum.</p>
<div class="row"><span class="pill">Nearest-neighbor p<sub>z</sub> NEGF</span><span class="pill">Conserved transverse momentum</span><span class="pill">Live calculation · works offline</span></div>
<section class="card">
<div class="row between"><h2>1. Set the device</h2><div class="inline-control"><label for="preset">Start from</label><select id="preset"><option value="npn">n–p–n junction</option><option value="uniform">Uniform graphene</option><option value="nnn">n–n–n barrier</option><option value="neutral">Neutral channel / evanescent modes</option><option value="asymmetric">Asymmetric source and drain</option><option value="bound">Closed leads / channel states</option></select><button id="reset">Reset</button></div></div>
<p class="small">All three controls are <b>electron energies in eV</b>, not applied voltages or reservoir chemical potentials. Sliders recalculate when released.</p>
<div class="controls">
<div class="control"><label for="eds">Source Dirac point <output id="edsValue"></output></label><input id="eds" type="range" min="-.35" max=".35" step=".005"></div>
<div class="control"><label for="edc">Channel Dirac point <output id="edcValue"></output></label><input id="edc" type="range" min="-.35" max=".35" step=".005"></div>
<div class="control"><label for="edd">Drain Dirac point <output id="eddValue"></output></label><input id="edd" type="range" min="-.35" max=".35" step=".005"></div>
<div class="control"><label for="length">Channel length <output id="lengthValue"></output></label><input id="length" type="range" min="5" max="120" step="1"></div>
<div class="control"><label for="ky"><span>k<sub>y</sub> from the valley</span> <output id="kyValue"></output></label><input id="ky" type="range" min="-.5" max=".5" step=".005"></div>
<div class="control"><label for="energy"><span>Probe energy E<sub>0</sub></span> <output id="energyValue"></output></label><input id="energy" type="range" min="-.45" max=".45" step=".002"></div>
</div>
<div class="row" style="margin-top:12px">
<div class="inline-control"><label for="valley">Valley</label><select id="valley"><option value="1">K</option><option value="-1">K′</option></select></div>
<div class="inline-control"><label for="eta">LDOS broadening η</label><select id="eta"><option value="0.0005">0.5 meV</option><option value="0.001" selected>1 meV</option><option value="0.002">2 meV</option><option value="0.005">5 meV</option></select></div>
<label class="inline-control"><input id="log" type="checkbox" checked>Log LDOS color scale</label>
<label class="inline-control"><input id="edges" type="checkbox" checked>Show mode thresholds</label>
</div>
<div class="stats"><div class="stat"><span>Actual channel length</span><strong id="lengthStat"></strong></div><div class="stat"><span>Selected mode gap</span><strong id="gapStat"></strong></div><div class="stat"><span>T at probe energy</span><strong id="tStat"></strong></div><div class="stat"><span>Band character at E₀</span><strong id="typeStat"></strong></div></div>
<div id="status" class="status" role="status">Preparing the NEGF calculation…</div><div id="modeStatus" class="callout"></div>
</section>
<div class="grid2">
<section class="card"><h2>2. Dirac-point profile</h2><p class="small">The solid line is E<sub>D</sub>(x); dashed lines delimit this k<sub>y</sub> mode. Vertical guides mark x = 0 and x = L.</p><canvas id="profile" class="plot" role="img" aria-label="Dirac-point energies versus position"></canvas></section>
<section class="card"><h2>3. Band slices at the selected k<sub>y</sub></h2><p class="small">Solid / dashed / dotted curves: source / channel / drain. Both bands of the actual NEGF Hamiltonian are shown.</p><canvas id="bands" class="plot" role="img" aria-label="Source channel drain fixed transverse momentum band structures"></canvas><div id="bandLegend" class="legend"></div></section>
</div>
<section class="card"><div class="row between"><h2>4. Energy-resolved transmission</h2><button id="exportT">Export transmission CSV</button></div><p class="small">Each curve is a single transverse mode, per spin and valley. A closed source or drain mode has T = 0; an evanescent channel can still transmit.</p><canvas id="transmission" class="plot wideplot" role="img" aria-label="Transmission versus energy for different transverse momenta"></canvas><div id="transLegend" class="legend"></div></section>
<section class="card"><div class="row between"><h2>5. Energy-resolved local density of states</h2><button id="exportRho">Export LDOS CSV</button></div><p class="small">Cell-averaged ρ(x,E;k<sub>y</sub>), in states/(eV nm), for one spin and one valley mode. This is available spectral weight—not occupied carrier density.</p><canvas id="ldos" class="plot ldosplot" role="img" aria-label="Local density of states as a function of position and energy"></canvas>
<p class="small">The fixed color range is 10<sup>−3</sup> to 5 states/(eV nm) in log view, or 0 to 5 in linear view. Values above 5 saturate visually but are preserved in the CSV. The 2 meV energy grid may miss very narrow peaks; η controls spectral resolution, not physical scattering.</p>
<div class="energyrow"><button id="play" class="primary">▶ Sweep energy</button><label for="energy2" class="small">E<sub>0</sub></label><input id="energy2" type="range" min="-.45" max=".45" step=".002" aria-label="Probe energy above the spatial LDOS slice"><output id="energy2Value"></output></div>
<canvas id="slice" class="plot wideplot" role="img" aria-label="Spatial local density of states at the selected probe energy"></canvas><p class="small">This line cut is recalculated at the exact E<sub>0</sub>, rather than taken from the nearest map pixel. The two probe-energy sliders are synchronized.</p>
</section>
<section class="card"><h2>Read the physics</h2>
<p id="insight"></p>
<details><summary>What is—and is not—in the model?</summary><p>We solve the graphene nearest-neighbor mode-space Hamiltonian using retarded semi-infinite lead self-energies, a recursive Green function, the Caroli transmission, and −Im G<sup>r</sup><sub>ii</sub>/π. The low-energy bands approach E<sub>D</sub> ± ℏv<sub>F</sub>√(q<sub>x</sub>²+k<sub>y</sub>²). No central-difference Dirac discretization or wide-band contacts are used.</p><p>The channel is phase coherent across both interfaces. There is no self-consistent Poisson solution, disorder, phonons, edge roughness, or reservoir bias. Changing the three Dirac energies does not itself impose different reservoir occupations. The source and drain are semi-infinite; the plotted ~12 nm buffers only show their near-interface LDOS.</p><p>The finite-k<sub>y</sub> gap is a projected mode gap, not a mass gap in the two-dimensional sheet. The near-normal-incidence transmission approaches the ideal Dirac value 1 at low energy, but finite-energy lattice corrections are retained. The default hopping is 2.7 eV and C–C distance is 0.142 nm, giving v<sub>F</sub> ≈ 0.874 × 10<sup>6</sup> m/s. Transport uses η = 10<sup>−9</sup> eV independently of the LDOS setting.</p></details>
<details><summary>Three quick experiments</summary><p><b>Uniform sheet:</b> select Uniform graphene. T should be nearly one wherever both leads have a propagating mode.</p><p><b>Interference:</b> select n–p–n. Increase L and compare the energy spacing of transmission peaks for an oblique mode.</p><p><b>Spectral weight without transmission:</b> select Closed leads / channel states. Bound channel states can appear in the broadened LDOS even while neither reservoir carries that transverse mode.</p></details>
<p class="footer">Primary method: T. Low et al., <i>Conductance asymmetry of graphene pn junction</i>, IEEE Trans. Electron Devices 56, 1292 (2009), <a href="https://arxiv.org/abs/0811.1295" target="_blank" rel="noopener">arXiv:0811.1295</a>, Sec. III. Unlike its single-junction conductance treatment, this laboratory retains the coherent finite channel. Related continuum results: <a href="https://arxiv.org/abs/cond-mat/0604323" target="_blank" rel="noopener">Katsnelson et al. (2006)</a>; <a href="https://arxiv.org/abs/cond-mat/0603315" target="_blank" rel="noopener">Tworzydło et al. (2006)</a>. Full derivations, Python source, and numerical tests accompany the notebook.</p>
</section>
</main><script>
__ENGINE__
const DEFAULTS=__DEFAULTS__;
const PALETTE=__PALETTE__;
const LUT=__LUT__;
const EGRID=Float64Array.from({length:451},(_,j)=>-.45+.002*j);
const REF_KY=[0,.12,.24];
const $=id=>document.getElementById(id);
let p={...DEFAULTS}, selectedKy=.18, probeE=0, spectral=null, curves=[], geomKey='', mapKey='', curveCache=new Map(), timer=null;
const FMT=(x,n=3)=>(Math.abs(x)<Math.pow(10,-n)/2?0:x).toFixed(n);
function getParams(){p={...DEFAULTS,dirac_source_eV:+$('eds').value,dirac_channel_eV:+$('edc').value,dirac_drain_eV:+$('edd').value,channel_length_nm:+$('length').value,valley:+$('valley').value,eta_ldos_eV:+$('eta').value};selectedKy=+$('ky').value;probeE=+$('energy').value;}
function setControls(){[['eds',p.dirac_source_eV],['edc',p.dirac_channel_eV],['edd',p.dirac_drain_eV],['length',p.channel_length_nm],['ky',selectedKy],['energy',probeE],['energy2',probeE],['valley',p.valley],['eta',p.eta_ldos_eV]].forEach(([k,v])=>$(k).value=v);labels();}
function labels(){for(const k of ['eds','edc','edd'])$(k+'Value').textContent=FMT(+$(k).value)+' eV';$('lengthValue').textContent=FMT(+$('length').value,0)+' nm';$('kyValue').textContent=FMT(+$('ky').value)+' nm⁻¹';$('energyValue').textContent=FMT(probeE)+' eV';$('energy2Value').textContent=FMT(probeE)+' eV';}
function preset(name){stop();p={...DEFAULTS};selectedKy=.18;probeE=0;
 if(name==='uniform'){p.dirac_channel_eV=-.2;}
 if(name==='nnn'){p.dirac_channel_eV=-.08;selectedKy=.12;}
 if(name==='neutral'){p.dirac_channel_eV=0;selectedKy=.06;}
 if(name==='asymmetric'){p.dirac_drain_eV=.05;selectedKy=.18;}
 if(name==='bound'){p.dirac_source_eV=0;p.dirac_channel_eV=-.2;p.dirac_drain_eV=0;selectedKy=.24;}
 setControls();schedule();}
function mkFrame(id,xmin,xmax,ymin,ymax,xlabel,ylabel,right=20){
 const canvas=$(id),r=canvas.getBoundingClientRect(),w=Math.max(290,r.width),h=r.height,dpr=window.devicePixelRatio||1;
 canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr);const c=canvas.getContext('2d');c.scale(dpr,dpr);c.clearRect(0,0,w,h);
 const a={c,w,h,l:65,r:w-right,t:16,b:h-51,xmin,xmax,ymin,ymax};a.X=x=>a.l+(x-xmin)/(xmax-xmin)*(a.r-a.l);a.Y=y=>a.b-(y-ymin)/(ymax-ymin)*(a.b-a.t);
 c.font='12px system-ui';c.fillStyle='#344955';c.strokeStyle='#dfe5e8';c.lineWidth=1;
 for(let j=0;j<=4;j++){const v=xmin+(xmax-xmin)*j/4,xx=a.X(v);c.beginPath();c.moveTo(xx,a.t);c.lineTo(xx,a.b);c.stroke();c.textAlign='center';c.fillText(Math.abs(v)>=10?FMT(v,0):FMT(v,2),xx,a.b+20);}
 for(let j=0;j<=4;j++){const v=ymin+(ymax-ymin)*j/4,yy=a.Y(v);c.beginPath();c.moveTo(a.l,yy);c.lineTo(a.r,yy);c.stroke();c.textAlign='right';c.fillText(FMT(v,2),a.l-9,yy+4);}
 c.strokeStyle='#93a6b0';c.strokeRect(a.l,a.t,a.r-a.l,a.b-a.t);c.fillStyle='#183041';c.font='13px system-ui';c.textAlign='center';c.fillText(xlabel,(a.l+a.r)/2,h-10);c.save();c.translate(16,(a.t+a.b)/2);c.rotate(-Math.PI/2);c.fillText(ylabel,0,0);c.restore();return a;
}
function plotLine(a,x,y,index=0,dash=[],width=2){const c=a.c;c.save();c.beginPath();c.rect(a.l,a.t,a.r-a.l,a.b-a.t);c.clip();c.strokeStyle=PALETTE[index%PALETTE.length];c.lineWidth=width;c.setLineDash(dash);c.beginPath();let started=false;for(let j=0;j<x.length;j++){if(!Number.isFinite(y[j])){started=false;continue;}if(!started){c.moveTo(a.X(x[j]),a.Y(y[j]));started=true;}else c.lineTo(a.X(x[j]),a.Y(y[j]));}c.stroke();c.restore();}
function guide(a,value,vertical=true,dash=[4,4]){const c=a.c;c.save();c.strokeStyle='#647783';c.lineWidth=1;c.setLineDash(dash);c.beginPath();if(vertical){c.moveTo(a.X(value),a.t);c.lineTo(a.X(value),a.b);}else{c.moveTo(a.l,a.Y(value));c.lineTo(a.r,a.Y(value));}c.stroke();c.restore();}
function legend(id,labels){$(id).innerHTML=labels.map((s,j)=>'<span><i class="swatch" style="background:'+PALETTE[j%PALETTE.length]+'"></i>'+s+'</span>').join('');}
function modeLabel(U,gap){let e=probeE-U;if(Math.abs(e)<gap-1e-12)return 'evanescent';if(Math.abs(e)<1e-10)return 'Dirac crossing';return e>0?'conduction / propagating':'valence / propagating';}
function drawProfile(){const g=spectral.geometry;const a=mkFrame('profile',g.xmin,g.xmax,-.65,.65,'Position x (nm)','Energy (eV)');const xx=[g.xmin,0,0,g.L,g.L,g.xmax],u=[p.dirac_source_eV,p.dirac_source_eV,p.dirac_channel_eV,p.dirac_channel_eV,p.dirac_drain_eV,p.dirac_drain_eV];plotLine(a,xx,u,0,[],2.7);if($('edges').checked){plotLine(a,xx,u.map(v=>v+spectral.halfGap),1,[6,4],1.6);plotLine(a,xx,u.map(v=>v-spectral.halfGap),1,[6,4],1.6);}guide(a,probeE,false,[2,3]);guide(a,0);guide(a,g.L);const c=a.c;c.fillStyle='#344955';c.font='11px system-ui';c.textAlign='center';[[g.xmin/2,'S'],[g.L/2,'channel'],[(g.L+g.xmax)/2,'D']].forEach(([x,s])=>c.fillText(s,a.X(x),a.t+15));}
function drawBands(){const a=mkFrame('bands',-.8,.8,-.85,.85,'qₓ from the valley (nm⁻¹)','Energy (eV)');const q=Float64Array.from({length:241},(_,j)=>-.8+1.6*j/240),t=p.hopping_eV,b=GrapheneEngine.hopping(selectedKy,p),cell=spectral.geometry.a;const off=Array.from(q,x=>Math.sqrt((t-b)**2+4*t*b*Math.sin(x*cell/2)**2));[p.dirac_source_eV,p.dirac_channel_eV,p.dirac_drain_eV].forEach((u,j)=>{const dash=[[],[7,4],[2,4]][j];plotLine(a,q,off.map(v=>u+v),j,dash,2);plotLine(a,q,off.map(v=>u-v),j,dash,2);});guide(a,probeE,false);legend('bandLegend',['source','channel','drain']);}
function drawTransmission(){const a=mkFrame('transmission',-.45,.45,0,1.04,'Energy E (eV)','Transmission T');curves.forEach((r,j)=>plotLine(a,EGRID,r.T,j,j===3?[6,3]:[],j===3?2.5:1.6));guide(a,probeE,true,[3,3]);legend('transLegend',curves.map((r,j)=>'kᵧ = '+FMT(r.ky)+' nm⁻¹'+(j===3?' (selected)':'')));}
function drawLDOS(){const g=spectral.geometry,nE=EGRID.length,log=$('log').checked;const a=mkFrame('ldos',g.xmin,g.xmax,-.45,.45,'Position x (nm)','Energy E (eV)',91);const off=document.createElement('canvas');off.width=g.n;off.height=nE;const ctx=off.getContext('2d'),image=ctx.createImageData(g.n,nE),pixels=image.data;const lo=log?Math.log10(.001):0,hi=log?Math.log10(5):5;for(let ie=0;ie<nE;ie++)for(let j=0;j<g.n;j++){let v=Math.max(spectral.rho[ie*g.n+j],1e-300);v=log?Math.log10(v):v;const f=Math.max(0,Math.min(1,(v-lo)/(hi-lo))),rgb=LUT[Math.min(255,Math.floor(f*255))],idx=((nE-1-ie)*g.n+j)*4;pixels[idx]=rgb[0];pixels[idx+1]=rgb[1];pixels[idx+2]=rgb[2];pixels[idx+3]=255;}ctx.putImageData(image,0,0);a.c.imageSmoothingEnabled=false;a.c.drawImage(off,a.l,a.t,a.r-a.l,a.b-a.t);const c=a.c;const barx=a.r+16,bw=15;for(let i=0;i<256;i++){const rgb=LUT[i];c.fillStyle='rgb('+rgb.join(',')+')';c.fillRect(barx,a.b-(i+1)/256*(a.b-a.t),bw,(a.b-a.t)/256+1);}c.font='11px system-ui';c.fillStyle='#344955';c.textAlign='left';const ticks=log?[.001,.01,.1,1,5]:[0,1,2,3,4,5];ticks.forEach(v=>{const f=((log?Math.log10(v):v)-lo)/(hi-lo),yy=a.b-f*(a.b-a.t);c.fillText(String(v),barx+bw+6,yy+4);});if($('edges').checked){const xx=[g.xmin,0,0,g.L,g.L,g.xmax],u=[p.dirac_source_eV,p.dirac_source_eV,p.dirac_channel_eV,p.dirac_channel_eV,p.dirac_drain_eV,p.dirac_drain_eV];plotLine(a,xx,u.map(v=>v+spectral.halfGap),1,[7,4],1.8);plotLine(a,xx,u.map(v=>v-spectral.halfGap),1,[7,4],1.8);}guide(a,0,true,[2,3]);guide(a,g.L,true,[2,3]);guide(a,probeE,false,[2,2]);c.strokeStyle='#647783';c.strokeRect(a.l,a.t,a.r-a.l,a.b-a.t);}
function drawSlice(){const one=GrapheneEngine.solve([probeE],selectedKy,p,p.eta_ldos_eV,true),g=one.geometry;let max=.7;for(const v of one.rho)max=Math.max(max,v*1.12);const a=mkFrame('slice',g.xmin,g.xmax,0,max,'Position x (nm)','ρ at E₀ [states/(eV nm)]');plotLine(a,g.x,one.rho,3,[],2);guide(a,0);guide(a,g.L);}
function stats(){const g=spectral.geometry,gap=spectral.halfGap,T=GrapheneEngine.solve([probeE],selectedKy,p,p.eta_transmission_eV,false).T[0];$('lengthStat').textContent=FMT(g.L)+' nm';$('gapStat').textContent=FMT(2*gap*1000,1)+' meV';$('tStat').textContent=T<1e-4&&T>0?T.toExponential(3):FMT(T,5);const types=[p.dirac_source_eV,p.dirac_channel_eV,p.dirac_drain_eV].map(u=>Math.abs(probeE-u)<1e-9?'0':(probeE>u?'n':'p'));$('typeStat').textContent=types.join(' – ');const labels=[p.dirac_source_eV,p.dirac_channel_eV,p.dirac_drain_eV].map(u=>modeLabel(u,gap));$('modeStatus').textContent='At the probe energy: source '+labels[0]+'; channel '+labels[1]+'; drain '+labels[2]+'.';let msg='In a propagating channel, reflections from the two interfaces can form Fabry–Pérot resonances. Compare the peaks in transmission with the spatial standing-wave patterns.';if(labels[0]==='evanescent'||labels[2]==='evanescent')msg='A lead has no propagating state for this transverse momentum at E₀, so its contribution to two-terminal transmission is zero. Nonzero LDOS is not sufficient for current: the states must also connect to propagating reservoir modes.';else if(labels[1]==='evanescent')msg='The channel is evanescent for the selected transverse momentum, but it has finite length. Contact wavefunctions can penetrate from both sides and produce nonzero tunneling. Increase L to suppress their overlap.';else if(Math.abs(selectedKy)<.0001)msg='At normal incidence, the ideal low-energy Dirac model transmits perfectly through a scalar barrier. Here the nearest-neighbor lattice Hamiltonian retains small finite-energy corrections rather than forcing T to equal one.';$('insight').textContent=msg;}
function drawAll(){if(!spectral)return;drawProfile();drawBands();drawTransmission();drawLDOS();drawSlice();stats();}
function recompute(){getParams();labels();const t0=performance.now();try{const key=JSON.stringify({...p,eta_ldos_eV:0});if(key!==geomKey){curveCache.clear();geomKey=key;}const qs=[...REF_KY,selectedKy];curves=qs.map(q=>{const k=q.toFixed(6);if(!curveCache.has(k))curveCache.set(k,GrapheneEngine.solve(EGRID,q,p,p.eta_transmission_eV,false));return curveCache.get(k);});const mk=key+'|'+selectedKy+'|'+p.eta_ldos_eV;if(mk!==mapKey){spectral=GrapheneEngine.solve(EGRID,selectedKy,p,p.eta_ldos_eV,true);mapKey=mk;}drawAll();$('status').className='status';$('status').textContent='Calculated '+EGRID.length+' energies × '+spectral.geometry.n+' spatial cells; '+Math.round(performance.now()-t0)+' ms. Source and drain are semi-infinite. The actual channel length is snapped to complete A/B cells.';window.LAB_READY=true;}catch(err){$('status').className='status error';$('status').textContent=err.message;console.error(err);}}
let pending=null;function schedule(){if(pending)clearTimeout(pending);$('status').textContent='Calculating…';pending=setTimeout(()=>{pending=null;recompute();},30);}
function syncEnergy(value){probeE=+value;$('energy').value=probeE;$('energy2').value=probeE;labels();drawAll();}
function stop(){if(timer){clearInterval(timer);timer=null;}$('play').textContent='▶ Sweep energy';}
function exportCSV(kind){if(!spectral)return;let lines=['# Graphene ballistic transport; energies=eV; lengths=nm; ky=nm^-1','# '+JSON.stringify(p),'# selected_ky_per_nm='+selectedKy];if(kind==='T'){lines.push('energy_eV,'+curves.map((r,j)=>'T_ky_'+r.ky.toFixed(3)+(j===3?'_selected':'')).join(','));for(let i=0;i<EGRID.length;i++)lines.push(EGRID[i].toPrecision(10)+','+curves.map(r=>r.T[i].toPrecision(12)).join(','));}else{lines.push('# LDOS per spin and valley mode; states/(eV nm)');lines.push('energy_eV,x_nm,ldos_per_eV_nm');const g=spectral.geometry;for(let ie=0;ie<EGRID.length;ie++)for(let j=0;j<g.n;j++)lines.push(EGRID[ie].toPrecision(10)+','+g.x[j].toPrecision(10)+','+spectral.rho[ie*g.n+j].toPrecision(12));}const blob=new Blob([lines.join('\n')],{type:'text/csv;charset=utf-8'}),url=URL.createObjectURL(blob),a=document.createElement('a');a.href=url;a.download=kind==='T'?'graphene_transmission.csv':'graphene_ldos.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);}
for(const id of ['eds','edc','edd','length','ky']){$(id).addEventListener('input',labels);$(id).addEventListener('change',()=>{stop();schedule();});}
for(const id of ['valley','eta'])$(id).addEventListener('change',()=>{stop();schedule();});
$('energy').addEventListener('input',e=>{stop();syncEnergy(e.target.value);});$('energy2').addEventListener('input',e=>{stop();syncEnergy(e.target.value);});
$('log').addEventListener('change',drawLDOS);$('edges').addEventListener('change',()=>{drawProfile();drawLDOS();});$('preset').addEventListener('change',e=>preset(e.target.value));$('reset').addEventListener('click',()=>{$('preset').value='npn';preset('npn');});
$('play').addEventListener('click',()=>{if(timer){stop();return;}$('play').textContent='❚❚ Pause';timer=setInterval(()=>{const next=probeE+.006;syncEnergy(next>.45?-.45:+next.toFixed(3));},130);});$('exportT').addEventListener('click',()=>exportCSV('T'));$('exportRho').addEventListener('click',()=>exportCSV('rho'));
let resizeTimer=null;window.addEventListener('resize',()=>{clearTimeout(resizeTimer);resizeTimer=setTimeout(drawAll,120);});window.addEventListener('pagehide',stop);
window.GrapheneTransport={engine:GrapheneEngine,getState:()=>({p,selectedKy,probeE,spectral,curves})};setControls();schedule();
</script></body></html>

'''

In [ ]:
# Use Matplotlib's existing default color cycle and colormap in the browser, too.
# No external JavaScript packages or remote assets are needed.
def build_browser_lab(device, selected_ky=0.18, probe_energy=0.0):
    default_colors = mpl.rcParams["axes.prop_cycle"].by_key()["color"]
    default_cmap = mpl.colormaps[mpl.rcParams["image.cmap"]]
    lut = np.rint(default_cmap(np.linspace(0, 1, 256))[:, :3]*255).astype(int).tolist()
    text = (BROWSER_HTML_TEMPLATE
            .replace("__ENGINE__", BROWSER_ENGINE_JS)
            .replace("__DEFAULTS__", json.dumps(vars(device)))
            .replace("__PALETTE__", json.dumps(default_colors))
            .replace("__LUT__", json.dumps(lut)))
    # The source template's initial values are customized, not its preset logic.
    text = text.replace("selectedKy=.18, probeE=0, spectral=null",
                        f"selectedKy={float(selected_ky)}, probeE={float(probe_energy)}, spectral=null")
    return text

lab_html = build_browser_lab(device, selected_ky_per_nm, probe_energy_eV)
lab_path = Path("Graphene_Ballistic_Transport_Lab.html")
lab_path.write_text(lab_html, encoding="utf-8")
print(f"Standalone interactive laboratory saved to: {lab_path.resolve()}")
# srcdoc contains the complete app; it does not need a local file URL or server.
# Trust and rerun this cell if your notebook viewer blocks active HTML.
display(HTML('<iframe title="Graphene ballistic transport laboratory" '
             'style="width:100%;height:2100px;border:1px solid #dbe3e8;border-radius:10px" '
             'srcdoc="'+html.escape(lab_html, quote=True)+'"></iframe>'))

## 9. Predict → test → explain

**Uniform graphene.** Set all three Dirac energies equal. Before looking, identify which energies allow the chosen lead mode. Does the solver give nearly unit transmission there, regardless of the nominal channel length?

**Normal versus oblique incidence.** Compare $k_y=0$ with a nonzero mode in an n–p–n device. Why does the oblique mode reflect more strongly? Lower the energy scales to distinguish lattice corrections from the ideal Dirac limit.

**Finite-length interference.** Keep an oblique mode propagating in the channel and double $L$. Do the transmission resonances become more closely spaced? Relate their spacing approximately to $\pi\hbar|v_x|/L$, where $v_x$ is the longitudinal group velocity.

**An evanescent channel.** Set the probe energy at the channel Dirac point, with doped source and drain. Increase $|k_y|$ or $L$. Which change reduces the overlap between the two contacts' penetration tails?

**LDOS does not equal transmitted current.** Use the “Closed leads / channel states” preset. Find spectral peaks in the channel while the corresponding lead modes are closed. Explain why a nonzero local spectrum does not guarantee nonzero two-terminal transmission.

**Change the energy reference.** Shift all three Dirac-point energies and the probe energy by the same amount. Which physical quantities should remain unchanged? Use the Python functions to verify this without running into the browser's fixed plotting window.

### Keep these distinctions visible
A projected mode gap is not a full 2D graphene gap. A semi-infinite contact is not a hard wall. Ballistic motion and phase coherence are separate assumptions; both are adopted here. LDOS is not occupation. A Dirac-point energy shift is not a chemical-potential bias. Spectral broadening is not automatically a physical scattering mechanism.

## 10. Export numerical data from the Python calculation

The browser already exports transmission and LDOS as CSV. The helper below saves the current Python arrays together with their parameters in a compact NumPy archive. No fourfold degeneracy is silently applied to the saved mode-resolved quantities.

In [ ]:
def export_current_run(path="graphene_transport_data.npz"):
    np.savez_compressed(
        path,
        energy_eV=energy_grid_eV,
        x_nm=selected_result["x_nm"],
        ky_per_nm=selected_ky_per_nm,
        ldos_per_eV_nm=selected_result["ldos_per_eV_nm"],
        transmission_selected=selected_result["transmission"],
        comparison_ky_per_nm=np.array(list(transmission_curves)),
        transmission_curves=np.array(list(transmission_curves.values())),
        device_json=json.dumps(vars(device)),
        actual_length_nm=device.actual_length_nm,
    )
    return Path(path).resolve()

# Uncomment to write the current results:
# print(export_current_run())

### Reading and model provenance

[Low et al., arXiv:0811.1295](https://arxiv.org/abs/0811.1295) motivates the transverse-mode NEGF formulation. [Castro Neto et al., arXiv:0709.1163](https://arxiv.org/abs/0709.1163) provides background on graphene tight binding and its low-energy Dirac limit. The continuum barrier and finite-strip comparisons connect to [Katsnelson et al., arXiv:cond-mat/0604323](https://arxiv.org/abs/cond-mat/0604323) and [Tworzydło et al., arXiv:cond-mat/0603315](https://arxiv.org/abs/cond-mat/0603315).

All plots in this notebook are generated by its own calculations; no paper figures are copied. The uniform-lead quadratic, scalar recursion, LDOS normalization, and numerical tests are written out so the implementation can be inspected and modified.